<a id="top"></a>

# Lab L0709: validate tool calls (the application validates)

```yaml
title: "Lab L0709: validate tool calls (the application validates)"
keywords: validation, hallucinated tool call, dispatch, missing argument, invented tool, tool_calls, langchain, protocol, lab
estimated duration: 25
```

> **Lesson:** L07. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L07/objectives.md). Solutions: `L0709_lab_solutions.ipynb`. **Pure Python — no API key needed** (you validate crafted tool calls, the same offline approach as the L0708 demo's fallback cell).
>
> **After this lab you can:** write the validation step that catches a hallucinated tool call, and explain why the schema alone does not prevent one.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Write the validator](#2-problem-1--write-the-validator)
- [3. Problem 2 — Run it over every crafted call](#3-problem-2--run-it-over-every-crafted-call)
- [4. Problem 3 — The three outcomes (written)](#4-problem-3--the-three-outcomes-written)
- [5. Problem 4 — Why doesn't the schema stop a bad call? (written)](#5-problem-4--why-doesnt-the-schema-stop-a-bad-call-written)
- [6. Self-check](#6-self-check)

## 1. Setup

The `calculator` (which raises on non-arithmetic input), plus crafted tool calls — one good, three hallucinated — as the same `{"name", "args", "id"}` dicts LangChain surfaces in `AIMessage.tool_calls`.

In [1]:
from typing import Any


def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, or raise ValueError on non-arithmetic input.

    Use this whenever the user asks for the result of a calculation.
    """
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    return str(eval(expression))


# The tool is a plain typed function; bind_tools would derive its definition. Here
# we validate crafted calls offline, so we only need the function itself.
TOOLS = [calculator]


# Crafted tool calls a model might emit — the same {"name", "args", "id"} dicts
# LangChain surfaces in AIMessage.tool_calls.
CALLS: list[dict[str, Any]] = [
    {"name": "calculator", "args": {"expression": "47219 * 8803"}, "id": "c0"},  # good
    {"name": "calculator", "args": {"expression": "twenty + 1"}, "id": "c1"},  # non-numeric
    {"name": "calculator", "args": {}, "id": "c2"},  # missing arg
    {"name": "wikipedia", "args": {"query": "geese"}, "id": "c3"},  # invented tool
]
print(len(CALLS), "crafted calls ready")

4 crafted calls ready


[↑ Back to top](#top)

## 2. Problem 1 — Write the validator

Implement `validate_call(call)` for a tool call (`{"name", "args", "id"}` dict), returning an `"OK: ..."` string with the result for a good call, or a `"REJECTED: ..."` string for each failure: an unknown tool name, a missing `expression` argument, or a `calculator` `ValueError`. The application — not the model — is responsible for catching all three.

In [2]:
def validate_call(call: dict[str, Any]) -> str:
    """Validate a tool call ({"name", "args", "id"} dict) and run it, or reject it loudly."""
    if call["name"] != "calculator":
        return f"REJECTED: no such tool {call['name']!r} (the model invented it)"
    args = call["args"]
    if "expression" not in args:
        return f"REJECTED: missing required argument 'expression' (got {args!r})"
    try:
        return f"OK: calculator({args['expression']!r}) = {calculator(args['expression'])}"
    except ValueError as exc:
        return f"REJECTED: {exc}"


print(validate_call(CALLS[0]))  # expect OK with a number

OK: calculator('47219 * 8803') = 415668857


[↑ Back to top](#top)

## 3. Problem 2 — Run it over every crafted call

Loop over `CALLS` and print the verdict for each. Three of the four should be REJECTED.

In [3]:
for call in CALLS:
    print(validate_call(call))

OK: calculator('47219 * 8803') = 415668857
REJECTED: refusing to evaluate non-arithmetic expression: 'twenty + 1'
REJECTED: missing required argument 'expression' (got {})
REJECTED: no such tool 'wikipedia' (the model invented it)


[↑ Back to top](#top)

## 4. Problem 3 — The three outcomes (written)

Name the **three** observable outcomes of handing a model a single tool, in one line each.

**Answer:**
1. **Calls the tool** — the reply's `.tool_calls` has a well-formed entry (valid arguments); the model judged the tool useful.
2. **Answers without it** — `.tool_calls` is empty and the reply is text only; a registered tool is an option, not an obligation.
3. **Calls it with bad arguments** — `.tool_calls` has an entry that is malformed, missing a required argument, or names an invented tool; the application must catch it.

[↑ Back to top](#top)

## 5. Problem 4 — Why doesn't the schema stop a bad call? (written)

"The application validates; the model proposes." In a sentence or two: why does binding a tool (whose JSON-Schema definition LangChain derives from your function) *not* prevent the model from emitting malformed or invented arguments?

**Answer:** The schema is part of the *prompt* — it tells the model what shape a call *should* have, but the model still **samples** tokens to produce the call, and nothing enforces the schema at generation time. LangChain parses those tokens into a tidy `.tool_calls` dict, but a clean *shape* is not a valid *value*: the arguments inside can be wrong-typed, missing, or name a tool that doesn't exist. Nothing checks them until the application does. The definition is a contract about shape, not a guarantee about behavior.

[↑ Back to top](#top)

## 6. Self-check

Compare against `L0709_lab_solutions.ipynb`. Done when `validate_call` returns `OK` for the first call and `REJECTED` (with a useful reason) for the non-numeric, missing-arg, and invented-tool calls.

[↑ Back to top](#top)